# SmartBike — Normal-only IMU anomaly detection

Notebook này clone/pull source mới nhất, lấy `TrainingData.zip` từ Google Drive, train convolutional autoencoder và load lại best model để inference. `ABNORMAL_SUSPECTED_FALL` không đồng nghĩa tai nạn đã được xác nhận.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tuanvu118/IOT-Project.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/IOT-Project')
DRIVE_ROOT = Path('/content/drive/MyDrive/SmartBike')
DRIVE_DATASET_ZIP = DRIVE_ROOT / 'datasets' / 'TrainingData.zip'
DRIVE_DATASET_FOLDER = DRIVE_ROOT / 'datasets' / 'TrainingData'
LOCAL_DATASET_ROOT = Path('/content/smartbike_dataset')
RUN_DIR = DRIVE_ROOT / 'runs' / 'anomaly_autoencoder'
FORCE_REFRESH_DATASET = False

# Lần đầu nên smoke test; sau khi chạy thành công hãy đặt cả hai thành None.
MAX_FILES_PER_SPLIT = 20
EPOCH_OVERRIDE = 2


## 1. Đồng bộ source code từ GitHub
Chạy lại cell này sau mỗi lần push code mới. `pull --ff-only` sẽ không âm thầm ghi đè thay đổi local trong Colab.

In [ ]:
import os
import subprocess

def run(command):
    print('+', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), check=True)

if (PROJECT_DIR / '.git').exists():
    run(['git', '-C', PROJECT_DIR, 'fetch', 'origin', BRANCH])
    run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH])
    run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', BRANCH])
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
run(['git', 'status', '--short', '--branch'])

## 2. Cài dependency

In [ ]:
run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'])
run(['python', '-m', 'pip', 'install', '-r', PROJECT_DIR / 'AI' / 'requirements-colab.txt'])

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 3. Mount Drive và chuẩn bị dataset
Ưu tiên `MyDrive/SmartBike/datasets/TrainingData.zip`. Notebook giải nén vào `/content` để tránh đọc hơn 10.000 file nhỏ trực tiếp qua Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import zipfile

EXPECTED = {'cruise', 'traffic', 'fun', 'wait', 'overtake'}

def looks_like_training_data(path: Path) -> bool:
    return path.is_dir() and EXPECTED.issubset({item.name for item in path.iterdir() if item.is_dir()})

if FORCE_REFRESH_DATASET and LOCAL_DATASET_ROOT.exists():
    if str(LOCAL_DATASET_ROOT).startswith('/content/'):
        shutil.rmtree(LOCAL_DATASET_ROOT)

if not LOCAL_DATASET_ROOT.exists():
    LOCAL_DATASET_ROOT.mkdir(parents=True)
    if DRIVE_DATASET_ZIP.is_file():
        print('Extracting', DRIVE_DATASET_ZIP)
        with zipfile.ZipFile(DRIVE_DATASET_ZIP) as archive:
            archive.extractall(LOCAL_DATASET_ROOT)
    elif DRIVE_DATASET_FOLDER.is_dir():
        print('Copying Drive folder to local Colab storage (ZIP is faster next time)')
        shutil.copytree(DRIVE_DATASET_FOLDER, LOCAL_DATASET_ROOT / 'TrainingData', dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            f'Upload TrainingData.zip to {DRIVE_DATASET_ZIP} or the folder to {DRIVE_DATASET_FOLDER}'
        )

candidates = [LOCAL_DATASET_ROOT, LOCAL_DATASET_ROOT / 'TrainingData', *LOCAL_DATASET_ROOT.rglob('TrainingData')]
DATASET_PATH = next((path for path in candidates if looks_like_training_data(path)), None)
if DATASET_PATH is None:
    raise RuntimeError(f'Cannot locate a valid TrainingData directory below {LOCAL_DATASET_ROOT}')

csv_count = len(list(DATASET_PATH.rglob('*.csv')))
print('DATASET_PATH =', DATASET_PATH)
print('CSV files =', csv_count)
RUN_DIR.mkdir(parents=True, exist_ok=True)

## 4. Tạo config cho lần chạy
Smoke test dùng ít file và 2 epoch. Full training giữ nguyên config 60 epoch với EarlyStopping.

In [ ]:
import yaml

BASE_CONFIG = PROJECT_DIR / 'AI' / 'configs' / 'anomaly_autoencoder.yaml'
RUNTIME_CONFIG = Path('/content/anomaly_autoencoder_runtime.yaml')
config = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
if EPOCH_OVERRIDE is not None:
    config['training']['epochs'] = int(EPOCH_OVERRIDE)
config['artifacts']['output_dir'] = str(RUN_DIR)
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(RUNTIME_CONFIG.read_text())

## 5. Phân tích dataset và vẽ tín hiệu mẫu

In [ ]:
analysis_command = [
    'python', '-m', 'AI.src.analyze_dataset',
    '--dataset', DATASET_PATH,
    '--config', RUNTIME_CONFIG,
    '--output', RUN_DIR / 'dataset_report.json',
    '--plot-dir', RUN_DIR / 'plots',
]
run(analysis_command)

## 6. Train autoencoder
Threshold được chọn từ validation normal; test chỉ dùng đo false-positive rate.

In [ ]:
train_command = [
    'python', '-m', 'AI.src.train',
    '--config', RUNTIME_CONFIG,
    '--dataset', DATASET_PATH,
    '--output', RUN_DIR,
]
if MAX_FILES_PER_SPLIT is not None:
    train_command += ['--max-files-per-split', str(MAX_FILES_PER_SPLIT)]
run(train_command)

## 7. Xem history và test metrics

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

history = json.loads((RUN_DIR / 'history.json').read_text())
metrics = json.loads((RUN_DIR / 'test_metrics.json').read_text())
scores = pd.read_csv(RUN_DIR / 'test_window_scores.csv')

epoch_table = pd.DataFrame({
    'epoch': np.arange(1, len(history['loss']) + 1),
    'train_mse': history['loss'],
    'validation_mse': history['val_loss'],
})
epoch_table['generalization_gap'] = epoch_table['validation_mse'] - epoch_table['train_mse']
best_row = epoch_table.loc[epoch_table['validation_mse'].idxmin()]

false_positives = int(metrics['false_positive_windows'])
total_normal = int(metrics['window_count'])
correct_normal = total_normal - false_positives
normal_acceptance = correct_normal / total_normal if total_normal else float('nan')

print('=== KẾT QUẢ TRAINING ===')
print(f"Số epoch đã chạy: {len(epoch_table)}")
print(f"Epoch tốt nhất: {int(best_row['epoch'])}")
print(f"Train MSE tại epoch tốt nhất: {best_row['train_mse']:.6f}")
print(f"Validation MSE tốt nhất: {best_row['validation_mse']:.6f}")
print(f"Chênh lệch validation - train: {best_row['generalization_gap']:.6f}")
print('\n=== ĐÁNH GIÁ TRÊN TEST CHỈ CÓ DỮ LIỆU BÌNH THƯỜNG ===')
print(f"Tổng cửa sổ NORMAL: {total_normal:,}")
print(f"Nhận đúng là NORMAL: {correct_normal:,}")
print(f"Cảnh báo nhầm là bất thường: {false_positives:,}")
print(f"Tỷ lệ nhận đúng NORMAL: {normal_acceptance:.2%}")
print(f"False-positive rate: {float(metrics['false_positive_rate']):.2%}")
print(f"Ngưỡng anomaly: {float(metrics['threshold']):.6f}")
print('\nPrecision/Recall/F1 phát hiện ngã: CHƯA THỂ TÍNH vì chưa có test label ngã.')

display(epoch_table.style.format({
    'train_mse': '{:.6f}',
    'validation_mse': '{:.6f}',
    'generalization_gap': '{:.6f}',
}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epoch_table['epoch'], epoch_table['train_mse'], label='train')
axes[0].plot(epoch_table['epoch'], epoch_table['validation_mse'], label='validation')
axes[0].axvline(int(best_row['epoch']), color='black', linestyle='--', alpha=.5, label='best epoch')
axes[0].set(xlabel='Epoch', ylabel='Reconstruction MSE', title='Loss theo từng epoch')
axes[0].legend()
axes[0].grid(alpha=.3)
axes[1].hist(scores['anomaly_score'], bins=60, alpha=.8)
axes[1].axvline(float(metrics['threshold']), color='red', linestyle='--', label='threshold')
axes[1].set(xlabel='Anomaly score', ylabel='Số cửa sổ', title='Phân bố điểm test NORMAL')
axes[1].legend()
axes[1].grid(alpha=.3)
plt.tight_layout()
plt.show()

## 8. Load lại best model và inference một file test chưa thấy

In [ ]:
manifest = json.loads((RUN_DIR / 'split_manifest.json').read_text())
unseen_csv = DATASET_PATH / manifest['test'][0]['path']
print('Unseen CSV:', unseen_csv)
run([
    'python', '-m', 'AI.src.inference',
    '--run', RUN_DIR,
    '--csv', unseen_csv,
    '--output', RUN_DIR / 'inference_example.csv',
])

## Sau smoke test
Nếu toàn bộ notebook chạy thành công, đổi `MAX_FILES_PER_SPLIT = None`, `EPOCH_OVERRIDE = None`, rồi chạy lại từ cell config. Artifact vẫn lưu trong Drive. Nếu muốn tiếp tục từ last checkpoint, thêm `--resume` vào `train_command`.